# Transform Results Data
1. Read bronze `results` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `results` table


#### Entity Relationship Diagram - Formula1 Bronze Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
bronze_table = f"{catalogo}.{esquema_bronze}.results"
silver_table = f"{catalogo}.{esquema_silver}.results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 & 4 - Read bronze `results` table, select only the required columns and standardise column names

In [0]:
results_df = (
  spark.table(bronze_table)
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingesta_timestamp",
                "archivo_fuente")
       .withColumnsRenamed({
            "constructorId": "id_constructor",
            "round": "numero_carrera",
             "points": "puntos",
            "status" :"estado",
            "season": "temporada",
            "driverId": "id_piloto",
            "raceName": "nombre_carrera",
            "date": "fecha_carrera",
            "grid": "posicion_grilla",
            "laps": "vueltas_completadas",
            "number": "numero_tarjeta",
            "position": "posicion_final",
            "positionText": "posicion_final_texto"
        })
)

#### Step 5 & 6 Apply Data Quality Checks 
- Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
- Remove duplicate records

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("temporada").isNotNull() &
            F.col("numero_carrera").isNotNull() &
            F.col("id_constructor").isNotNull() &
            F.col("id_piloto").isNotNull() 
        )
        .dropDuplicates(["temporada", "numero_carrera", "id_constructor", "id_piloto"])
)

In [0]:
display(results_df.count() - results_valid_df.count())

In [0]:
#results_valid_df
from pyspark.sql.functions import when, col
df_replace_case =( results_valid_df.withColumn("estado", F.regexp_replace(F.col("estado"), "Laps", "Vueltas"))
                                   .withColumn("estado", F.regexp_replace(F.col("estado"), "Lap", "Vuelta"))
                                   .withColumn( "estado",
                                                 when(col("estado") =="Oil leak",  "Fuga de Aceite")
                                                .when(col("estado") =="Engine",  "Motor")
                                                .when(col("estado") =="Transmission",  "Caja de Velocidades")
                                                .when(col("estado") =="Gearbox",  "Caja de Cambios")
                                                .when(col("estado") =="Magneto",  "Magneto")
                                                .when(col("estado") =="Finished",  "Finalizado")
                                                .when(col("estado") =="Accident",  "Accidente")
                                                .when(col("estado") =="Collision",  "Choque")
                                                .when(col("estado") =="Safety Car",  "Seguridad")
                                                .when(col("estado") =="Safety car",  "Seguridad")
                                                .when(col("estado") =="Collision with wall",  "Choque con pared")
                                                .when(col("estado") =="Collision with barrier",  "Choque con barrera")
                                                .when(col("estado") =="Collision with car",  "Choque con coche")
                                                .when(col("estado") =="Collision with other car",  "Choque con otro coche")
                                                .when(col("estado") =="Collision with other vehicle",  "Choque con otro vehículo")
                                                .when(col("estado") =="Not classified",  "No clasificado")
                                                .when(col("estado")=="Water pressure",  "Presión de agua")
                                                .when(col("estado")=="Disqualified",  "Descalificado")
                                                .when(col("estado")=="Electrical",  "Falla Eléctrica")
                                                .when(col("estado")=="Hydraulics",  "Falla Hidráulica")
                                                .when(col("estado")=="Brakes",  "Frenos")
                                                .when(col("estado")=="Suspension",  "Suspensión")
                                                .when(col("estado")=="Tyre",  "Neumático")
                                                .when(col("estado")=="Tyres",  "Neumáticos")
                                                .when(col("estado")=="Wheel",  "Rueda")
                                                .when(col("estado")=="Wheel rim",  "Llanta")
                                                .when(col("estado")=="Fuel pressure",  "Presión de Combustible")
                                                .when(col("estado")=="Fuel system",  "Sistema de Combustible")
                                                .when(col("estado")=="Fuel leak",  "Fuga de Combustible")
                                                .when(col("estado")=="Overheating",  "Sobrecalentamiento")
                                                .when(col("estado")=="Radiator",  "Radiador")
                                                .when(col("estado")=="Throttle",  "Acelerador")
                                                .when(col("estado")=="Clutch",  "Embrague")
                                                .when(col("estado")=="Driveshaft",  "Árbol de Transmisión")
                                                .when(col("estado")=="Differential",  "Diferencial")
                                                .when(col("estado")=="Steering",  "Dirección")
                                                .when(col("estado")=="Spun off",  "Salida de Pista")
                                                .when(col("estado")=="Spin",  "Trompo")
                                                .when(col("estado")=="Withdrew",  "Retirado")
                                                .when(col("estado")=="Retired",  "Retirado")
                                                .when(col("estado")=="Mechanical",  "Falla Mecánica")
                                                .when(col("estado")=="Power Unit",  "Unidad de Potencia")
                                                .when(col("estado")=="Power unit",  "Unidad de Potencia")
                                                .when(col("estado")=="ERS",  "Sistema de Recuperación de Energía")
                                                .when(col("estado")=="Battery",  "Batería")
                                                .when(col("estado")=="Exhaust",  "Escape")
                                                .when(col("estado")=="Vibrations",  "Vibraciones")
                                                .when(col("estado")=="Fire",  "Incendio")
                                                .when(col("estado")=="Driver unwell",  "Piloto Indispuesto")
                                                
                                                .otherwise(col("estado"))
                                               )
                                     .withColumn( "posicion_final_texto",
                                                  when(col("posicion_final_texto") =="R",  "Retirado")
                                                  .when(col("posicion_final_texto") =="D",  "Descalificado")
                                                  .otherwise(col("posicion_final_texto"))
                                                 )
                                   
                                   )
                                   
                                    

#### Step 7 - Transform values of column `race_name` to Title Case

In [0]:
results_final_df = (
    df_replace_case
        .withColumn('nombre_carrera', F.initcap(F.col("nombre_carrera")))
)

#### Step 8 - Write the transformed data to silver `results` table

In [0]:
(
    results_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))